# 🤖 Clase 2 · La Revolución de los Transformers — Construyendo **MiniGPT**

**Módulo:** Científico de Datos e Inteligencia Artificial Aplicada · **Unidad 2:** NLP y el Auge de la IA Generativa (LLMs)

---

### 🎯 Objetivos de la clase
Al terminar podrás:
1. Explicar **por qué los Transformers reemplazaron a las RNN** y qué problema resuelve la *atención*.
2. Implementar **self-attention desde cero** (Q, K, V, softmax, máscara causal) con PyTorch.
3. Diferenciar arquitecturas **Encoder-only (BERT)** y **Decoder-only (GPT)** con ejemplos reales.
4. Entender **cómo genera texto un GPT**: predicción del siguiente token, temperatura, top-k y top-p.
5. Convertir un GPT-2 "base" en un **chatbot (MiniGPT)** mediante *fine-tuning supervisado* (SFT).

### 🧭 El proyecto del curso: MiniGPT
Durante la Unidad 2 construimos **nuestro propio "ChatGPT" en miniatura**:

| Clase | Qué le agregamos a MiniGPT |
|---|---|
| **Clase 2 (hoy)** | Entendemos el Transformer por dentro y entrenamos MiniGPT (GPT-2 en español) para que siga un formato de chat |
| Clase 3 | Lo convertimos en un asistente real con un LLM instruct (Qwen3), *prompt engineering*, memoria y streaming |

### ⏱️ Agenda (3 horas)
| Tiempo | Bloque |
|---|---|
| 0:00 – 0:15 | Intro: de embeddings a Transformers |
| 0:15 – 0:35 | **1.** Tokenización subword (BPE) |
| 0:35 – 1:15 | **2.** Self-attention y bloque Transformer desde cero |
| 1:15 – 1:30 | **3.** Mirando dentro de GPT-2 |
| 1:30 – 1:40 | ☕ Pausa |
| 1:40 – 2:00 | **4.** Encoder (BERT) vs Decoder (GPT) |
| 2:00 – 2:25 | **5.** Cómo genera texto un GPT (sampling) |
| 2:25 – 2:55 | **6.** MiniGPT: fine-tuning para chatear |
| 2:55 – 3:00 | Cierre y reto |

> ## 🧑‍🏫 Guía para el docente (no aparece en el notebook de estudiantes)
> - **Antes de la clase:** ejecuta la sección 0 completa con internet. Se descargan ~1 GB: `datificate/gpt2-small-spanish` (~500 MB) y `dccuchile/bert-base-spanish-wwm-cased` (~440 MB). Quedan en caché (`~/.cache/huggingface`) y en clase cargan en segundos.
> - Pide a los estudiantes que también ejecuten la sección 0 **antes** de llegar (o en los primeros 5 minutos mientras haces la intro).
> - **Con GPU NVIDIA** (ej. RTX 4070) el fine-tuning entrena el modelo completo, 25 épocas, en ~1–2 min. Si la celda de verificación dice que tu PyTorch es solo-CPU, instala la versión CUDA antes de clase.
> - **En CPU** (estudiantes sin GPU) congela 6 bloques y hace 15 épocas: ~6–12 min (estimado). Lánzalo y mientras entrena explica la celda de *máscara de labels*.
> - Para evaluar, `responder()` usa *greedy* (`muestrear=False`): así se ve lo que el modelo aprendió, sin el azar del muestreo.
> - Las celdas marcadas como **LIVE CODING** están vacías en el notebook de estudiantes. Las de configuración y gráficas vienen completas para no perder tiempo.
> - Si vas atrasado: la sección 3 y la parte de *multi-cabeza* de la sección 2 se pueden mostrar ejecutadas sin escribir el código.

## 📚 Teoría: ¿por qué aparecieron los Transformers?

En la Clase 1 convertimos texto en **vectores (embeddings)**. Pero un embedding clásico (Word2Vec o la capa `Embedding` de Keras) le da **el mismo vector a una palabra sin importar el contexto**:

- "Me senté en el **banco** del parque"
- "El **banco** me cobró una comisión"

Para resolverlo hace falta un modelo que **lea la frase completa** y ajuste el significado de cada palabra según las demás.

**Antes de 2017: Redes Recurrentes (RNN / LSTM)**
- Leen el texto **palabra por palabra, en orden** → no se pueden paralelizar (lentas de entrenar en GPU).
- La información del inicio se va "diluyendo" en frases largas (*memoria corta*, gradientes que se desvanecen).

**2017: "Attention Is All You Need" (Vaswani et al., Google)**
- Elimina la recurrencia: **cada palabra mira directamente a todas las demás** con el mecanismo de **atención**.
- Todo se calcula con multiplicaciones de matrices → **paralelizable** → se puede entrenar con muchísimos más datos.

**Línea de tiempo**
| Año | Hito |
|---|---|
| 2017 | Transformer (traducción automática) |
| 2018 | **BERT** (Google, encoder) y **GPT-1** (OpenAI, decoder) |
| 2019 | **GPT-2** (1.500 M parámetros) — el que usaremos hoy, versión pequeña en español |
| 2020 | GPT-3 (175.000 M parámetros) |
| 2022 | ChatGPT: GPT + ajuste con instrucciones + RLHF |
| 2023–hoy | LLMs abiertos: Llama, Mistral, Qwen, Gemma… |

**Idea clave para la clase:** un LLM como ChatGPT o Claude es, en el fondo, **un Transformer decoder que predice el siguiente token**, entrenado a gran escala y luego ajustado para conversar. Hoy construimos cada pieza en pequeño.

---
## 0. Preparación del entorno
Instalamos lo que falte **en el mismo Python que usa este kernel**, importamos librerías y fijamos la configuración.

In [ ]:
# Instala lo que falte en el MISMO Python que usa este kernel
import importlib.util, subprocess, sys

paquetes = {"torch": "torch", "transformers": "transformers", "pandas": "pandas",
            "matplotlib": "matplotlib"}
faltan = [pip for modulo, pip in paquetes.items() if importlib.util.find_spec(modulo) is None]

print("Python del kernel:", sys.executable)
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
    print("✓ Listo. Si algún import falla, reinicia el kernel.")
else:
    print("✓ Todo instalado.")

In [ ]:
import json
import math
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
    get_linear_schedule_with_warmup,
    pipeline,
)

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 120)

SEED = 42
def fijar_semilla(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

fijar_semilla()

# Dispositivo: GPU NVIDIA (cuda), GPU Apple (mps) o CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"torch {torch.__version__} | transformers {transformers.__version__} | device: {device}")

In [ ]:
# ¿Tienes GPU NVIDIA pero torch no la ve? (pasa si instalaste la versión "+cpu" de PyTorch)
import shutil
if device.type == "cuda":
    print("🚀 Usando GPU:", torch.cuda.get_device_name(0))
elif shutil.which("nvidia-smi"):
    print("⚠️ Tu computador tiene GPU NVIDIA, pero tu PyTorch es la versión solo-CPU:", torch.__version__)
    print("   Instala la versión con CUDA (en una terminal con tu entorno activado) y reinicia el kernel:")
    print("   pip uninstall -y torch")
    print("   pip install torch --index-url https://download.pytorch.org/whl/cu128")
    print("   (si ese comando falla, copia el de https://pytorch.org/get-started/locally/ eligiendo pip + CUDA)")
else:
    print("💻 Sin GPU NVIDIA: entrenaremos en CPU congelando capas (más lento, pero funciona).")

In [ ]:
# ---------------- CONFIGURACIÓN DE LA CLASE ----------------
MODELO_GPT  = "datificate/gpt2-small-spanish"          # GPT-2 small (124 M) preentrenado en Wikipedia en español
MODELO_BERT = "dccuchile/bert-base-spanish-wwm-cased"  # BETO: BERT en español (U. de Chile)

RUTA_DATOS = Path("..") / "datasets" / "dialogos_minigpt.jsonl"
DIR_SALIDA = Path("..") / "modelo" / "miniGPT-es"      # aquí guardamos MiniGPT (lo usamos en la Clase 3)

# Formato de chat que aprenderá MiniGPT
SYSTEM_PROMPT = "Eres MiniGPT, un asistente que responde en español de forma breve y clara."
TAG_USUARIO = "### Usuario:"
TAG_BOT = "### MiniGPT:"

# Hiperparámetros del fine-tuning (pensados para CPU)
CAPAS_CONGELADAS = 6 if device.type == "cpu" else 0   # de 12 bloques; en GPU entrenamos el modelo completo
EPOCHS = 15 if device.type == "cpu" else 25           # con GPU (ej. RTX 4070) 25 épocas tardan ~1-2 min
LEARNING_RATE = 3e-4 if CAPAS_CONGELADAS else 1e-4
BATCH_SIZE = 8

print("Dataset existe:", RUTA_DATOS.exists(), "→", RUTA_DATOS.resolve())

---
## 1. Tokenización subword (BPE): cómo "lee" un GPT

En la Clase 1 tokenizamos por **palabras** (`Tokenizer` de Keras) y vimos el problema de las palabras desconocidas (`<OOV>`).
Los LLMs usan **subpalabras**: el algoritmo **BPE (Byte Pair Encoding)** empieza con caracteres y va **fusionando los pares más frecuentes**.

- Palabras comunes → **1 token** (`casa`, `modelo`)
- Palabras raras → **varios tokens** (`desoxirribonucleico` → `des` + `oxi` + …)
- **Nunca hay OOV**: en el peor caso se parte hasta llegar a bytes.

> GPT-2 trabaja con **bytes**: en la vista "interna" el símbolo **`Ġ`** significa "aquí había un espacio" y las tildes aparecen raras (`Ã³` = `ó`) porque en UTF-8 la `ó` ocupa 2 bytes. Con `tokenizer.decode` lo vemos normal.

In [ ]:
# 1.1 Cargamos el tokenizer de GPT-2 en español y tokenizamos una frase
tokenizer = AutoTokenizer.from_pretrained(MODELO_GPT)

frase = "Los transformers revolucionaron el procesamiento de lenguaje natural."
ids = tokenizer.encode(frase)
tokens = tokenizer.convert_ids_to_tokens(ids)

print("Tamaño del vocabulario:", tokenizer.vocab_size)
pd.DataFrame({"id": ids,
              "token interno": tokens,                                   # bytes: 'Ġ' = espacio, 'Ã³' = 'ó'
              "texto": [tokenizer.decode([i]) for i in ids]}).T

In [ ]:
# 1.2 Palabras vs tokens: las palabras raras se parten en varios pedazos
ejemplos = [
    "Hola, ¿cómo estás?",
    "La inteligencia artificial aprende de los datos.",
    "El ácido desoxirribonucleico contiene información genética.",
    "Supercalifragilisticoespialidoso",
    "jajajaja 😂😂",
]
filas = []
for texto in ejemplos:
    ids_t = tokenizer.encode(texto)
    filas.append({"texto": texto, "palabras": len(texto.split()), "tokens": len(ids_t),
                  "tokens (vista)": " | ".join(tokenizer.decode([i]) for i in ids_t)})
pd.DataFrame(filas)

### 📚 Teoría: ¿por qué subpalabras y no palabras o caracteres?
| Unidad | Vocabulario | Problema |
|---|---|---|
| Palabra | Enorme (cientos de miles) | Palabras nuevas = `<OOV>` |
| Carácter | Minúsculo (~100) | Secuencias larguísimas, cada token aporta poco significado |
| **Subpalabra (BPE)** | Mediano (≈ 50.000) | **Equilibrio**: sin OOV y secuencias razonables |

**Cómo se entrena BPE (intuición):**
1. Empieza con el vocabulario de caracteres (o bytes).
2. Cuenta qué par de símbolos aparece junto más veces en el corpus (ej. `e` + `s`).
3. Lo fusiona en un nuevo símbolo `es` y lo agrega al vocabulario.
4. Repite hasta llegar al tamaño de vocabulario deseado.

**Consecuencias prácticas que deben saber los estudiantes:**
- Los LLMs cobran y limitan su memoria **en tokens, no en palabras** (en español ≈ 1,3–1,6 tokens por palabra).
- Por eso a los LLMs les cuesta **contar letras** ("¿cuántas r tiene *ferrocarril*?"): nunca ven letras, ven tokens.
- Un tokenizer entrenado en inglés parte más las palabras en español → más tokens → más caro y más lento. Por eso usamos un GPT-2 **entrenado en español**.

---
## 2. Self-Attention desde cero

**Problema:** el embedding de "banco" debería cambiar según el contexto. **Solución:** que cada token **mire a los demás** y mezcle su información según qué tan relevantes sean.

Cada token se proyecta en tres vectores (analogía con una búsqueda en una biblioteca):
| Vector | Pregunta que responde | Analogía |
|---|---|---|
| **Query (Q)** | ¿Qué estoy buscando? | Lo que escribes en el buscador |
| **Key (K)** | ¿Qué ofrezco yo? | La etiqueta/título de cada libro |
| **Value (V)** | ¿Qué información aporto? | El contenido del libro |

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

1. $QK^\top$ → **puntaje de afinidad** entre cada par de tokens.
2. $/\sqrt{d_k}$ → escalado para que el softmax no se sature.
3. **softmax** → cada fila se convierte en **pesos que suman 1** ("cuánta atención le pongo a cada token").
4. $\times V$ → cada token sale como una **mezcla ponderada** de la información de los demás.

In [ ]:
# 2.1 Una frase de juguete: 7 tokens con embeddings aleatorios de dimensión 16
torch.manual_seed(0)
tokens_toy = ["El", "gato", "se", "sentó", "en", "la", "alfombra"]
T, d_model = len(tokens_toy), 16

x = torch.randn(T, d_model)            # [T, d_model]  (en un modelo real vienen de la capa de embeddings)

# Tres proyecciones lineales aprendibles: W_q, W_k, W_v
d_k = 16
W_q = nn.Linear(d_model, d_k, bias=False)
W_k = nn.Linear(d_model, d_k, bias=False)
W_v = nn.Linear(d_model, d_k, bias=False)

Q, K, V = W_q(x), W_k(x), W_v(x)
print("x:", tuple(x.shape), "| Q:", tuple(Q.shape), "| K:", tuple(K.shape), "| V:", tuple(V.shape))

In [ ]:
# 2.2 La fórmula de la atención paso a paso
puntajes = Q @ K.T / math.sqrt(d_k)          # [T, T] afinidad de cada token con cada token
pesos = F.softmax(puntajes, dim=-1)          # cada fila suma 1
salida = pesos @ V                           # [T, d_k] nueva representación "contextual"

print("Puntajes:", tuple(puntajes.shape), "| Pesos:", tuple(pesos.shape), "| Salida:", tuple(salida.shape))
print("¿Cada fila suma 1?", torch.allclose(pesos.sum(dim=-1), torch.ones(T)))

In [ ]:
# Función auxiliar para dibujar mapas de atención (viene lista)
def plot_atencion(pesos, tokens, titulo="Mapa de atención", ax=None):
    pesos = pesos.detach().cpu().numpy() if torch.is_tensor(pesos) else np.asarray(pesos)
    if ax is None:
        _, ax = plt.subplots(figsize=(0.6 * len(tokens) + 2, 0.6 * len(tokens) + 1.5))
    ax.imshow(pesos, cmap="Purples", vmin=0, vmax=max(pesos.max(), 1e-9))
    ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set_xlabel("Key (token al que se mira)")
    ax.set_ylabel("Query (token que mira)")
    ax.set_title(titulo)
    if len(tokens) <= 12:
        for i in range(len(tokens)):
            for j in range(len(tokens)):
                v = pesos[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if v > 0.5 * pesos.max() else "black")
    return ax

plot_atencion(pesos, tokens_toy, "Self-attention SIN máscara (estilo BERT): todos ven a todos")
plt.tight_layout(); plt.show()

### 🎭 La máscara causal: la diferencia entre BERT y GPT
Un modelo que **genera** texto no puede "hacer trampa" mirando las palabras futuras (¡todavía no existen!).
La **máscara causal** pone $-\infty$ en los puntajes de los tokens futuros → después del softmax su peso es **0**.
El mapa queda **triangular**: el token *i* solo ve a los tokens 0…*i*.

In [ ]:
# 2.3 Máscara causal: cada token solo puede mirar hacia atrás
mascara = torch.triu(torch.ones(T, T), diagonal=1).bool()     # True arriba de la diagonal = futuro
puntajes_causales = puntajes.masked_fill(mascara, float("-inf"))
pesos_causales = F.softmax(puntajes_causales, dim=-1)

print(mascara.int())
plot_atencion(pesos_causales, tokens_toy, "Self-attention CON máscara causal (estilo GPT)")
plt.tight_layout(); plt.show()

### 📚 Teoría: ¿por qué dividir entre $\sqrt{d_k}$?
Si Q y K tienen componentes con varianza ≈ 1, el producto punto de dos vectores de dimensión $d_k$ tiene varianza ≈ $d_k$.
Con $d_k = 64$ los puntajes pueden valer ±20 → el **softmax se satura** (un token recibe ≈ 100% del peso y el resto ≈ 0) → los gradientes casi desaparecen y el modelo aprende mal.
Dividir entre $\sqrt{d_k}$ devuelve la varianza a ≈ 1. La siguiente celda lo demuestra numéricamente.

In [ ]:
# 2.4 (Demostración) Efecto del escalado en el softmax con d_k grande
d_grande = 512
q, k = torch.randn(10, d_grande), torch.randn(10, d_grande)
sin_escala = F.softmax(q @ k.T, dim=-1)
con_escala = F.softmax(q @ k.T / math.sqrt(d_grande), dim=-1)
print(f"Peso máximo promedio SIN escalar: {sin_escala.max(dim=-1).values.mean():.3f}  ← casi one-hot (saturado)")
print(f"Peso máximo promedio CON escala : {con_escala.max(dim=-1).values.mean():.3f}  ← reparte la atención")

### 🧠 Atención multi-cabeza y el bloque Transformer
- **Multi-cabeza:** en vez de una sola atención, usamos *h* atenciones pequeñas en paralelo. Cada **cabeza** aprende un tipo de relación distinta (sintaxis, correferencia, posición…). Al final se concatenan.
- **Bloque Transformer (estilo GPT-2):**

```
x ──► LayerNorm ──► Atención multi-cabeza (causal) ──► + (residual)
  ──► LayerNorm ──► Red feed-forward (MLP 4× más ancha, GELU) ──► + (residual) ──► siguiente bloque
```
- **Embeddings de posición:** la atención no sabe el orden de las palabras, así que se **suma** un vector por posición.
- **GPT completo** = embeddings (token + posición) → **N bloques** → LayerNorm → capa lineal que da un puntaje (**logit**) para cada token del vocabulario.

In [ ]:
# 2.5 Atención multi-cabeza como módulo de PyTorch
class AtencionMultiCabeza(nn.Module):
    def __init__(self, d_model, n_cabezas, causal=True):
        super().__init__()
        assert d_model % n_cabezas == 0, "d_model debe ser divisible entre n_cabezas"
        self.h, self.d_k, self.causal = n_cabezas, d_model // n_cabezas, causal
        self.qkv = nn.Linear(d_model, 3 * d_model)     # Q, K y V en una sola multiplicación
        self.salida = nn.Linear(d_model, d_model)

    def forward(self, x):                               # x: [B, T, d_model]
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=-1)
        # [B, T, D] -> [B, h, T, d_k]: cada cabeza trabaja con su propio pedazo del vector
        q, k, v = (t.view(B, T, self.h, self.d_k).transpose(1, 2) for t in (q, k, v))
        puntajes = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)        # [B, h, T, T]
        if self.causal:
            mascara = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            puntajes = puntajes.masked_fill(mascara, float("-inf"))
        pesos = F.softmax(puntajes, dim=-1)
        out = (pesos @ v).transpose(1, 2).contiguous().view(B, T, D)  # concatenar cabezas
        return self.salida(out), pesos

mha = AtencionMultiCabeza(d_model=16, n_cabezas=4)
out, pesos_mha = mha(x.unsqueeze(0))
print("Salida:", tuple(out.shape), "| Pesos por cabeza:", tuple(pesos_mha.shape))

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for c in range(4):
    plot_atencion(pesos_mha[0, c], tokens_toy, f"Cabeza {c}", ax=axes[c])
plt.tight_layout(); plt.show()

In [ ]:
# 2.6 Bloque Transformer y un "MiniTransformer" completo (arquitectura GPT, SIN entrenar)
class BloqueTransformer(nn.Module):
    def __init__(self, d_model, n_cabezas):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.atencion = AtencionMultiCabeza(d_model, n_cabezas, causal=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x):
        a, _ = self.atencion(self.ln1(x))
        x = x + a                       # conexión residual 1
        x = x + self.mlp(self.ln2(x))   # conexión residual 2
        return x

class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_cabezas=4, n_bloques=4, max_pos=256):
        super().__init__()
        self.emb_token = nn.Embedding(vocab_size, d_model)
        self.emb_pos = nn.Embedding(max_pos, d_model)
        self.bloques = nn.Sequential(*[BloqueTransformer(d_model, n_cabezas) for _ in range(n_bloques)])
        self.ln_final = nn.LayerNorm(d_model)
        self.cabeza_lm = nn.Linear(d_model, vocab_size, bias=False)   # un logit por token del vocabulario

    def forward(self, ids):                                   # ids: [B, T]
        posiciones = torch.arange(ids.shape[1], device=ids.device)
        x = self.emb_token(ids) + self.emb_pos(posiciones)    # significado + posición
        x = self.bloques(x)
        return self.cabeza_lm(self.ln_final(x))               # [B, T, vocab]

mini = MiniTransformer(vocab_size=len(tokenizer))
ids_prueba = torch.tensor([tokenizer.encode("La inteligencia artificial")])
logits = mini(ids_prueba)
n_param_mini = sum(p.numel() for p in mini.parameters())
print(f"Entrada {tuple(ids_prueba.shape)} → logits {tuple(logits.shape)}  (un puntaje por cada token del vocabulario)")
print(f"Parámetros del MiniTransformer: {n_param_mini:,}")

### 📚 Teoría: piezas del bloque que suelen generar preguntas
- **Conexiones residuales (`x + f(x)`):** dejan pasar la señal original "por un atajo". Permiten apilar decenas de bloques sin que el gradiente se pierda (la misma idea de ResNet en la clase de CNN).
- **LayerNorm:** normaliza cada vector (media 0, varianza 1) → entrenamiento estable. GPT-2 la aplica **antes** de cada sub-capa (*pre-LN*).
- **MLP / feed-forward:** la atención **mezcla** información entre tokens; el MLP **procesa** cada token por separado. Se cree que buena parte del "conocimiento factual" vive en estos MLP. Ocupan ~2/3 de los parámetros.
- **GELU:** activación suave parecida a ReLU (no corta en seco los negativos).
- **Embedding de posición aprendido (GPT-2)** vs **sinusoidal** (Transformer original) vs **RoPE** (Llama, Qwen: rota Q y K según la posición; generaliza mejor a textos largos).
- Nuestro `MiniTransformer` tiene la **misma arquitectura** que GPT-2; la diferencia es el tamaño y que **no está entrenado** (sus predicciones serían aleatorias).

---
## 3. Mirando dentro de GPT-2 real
Ahora cargamos **GPT-2 small en español** (preentrenado leyendo Wikipedia) y comparamos su tamaño con nuestro MiniTransformer.
Pedimos `output_attentions=True` para **ver los mapas de atención reales** que aprendió.

In [ ]:
# 3.1 Cargar GPT-2 en español (attn_implementation="eager" para poder devolver los pesos de atención)
inicio = time.time()
modelo = AutoModelForCausalLM.from_pretrained(MODELO_GPT, attn_implementation="eager").to(device)
modelo.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

cfg = modelo.config
n_param_gpt = sum(p.numel() for p in modelo.parameters())
print(f"Cargado en {time.time() - inicio:.1f}s")

pd.DataFrame({
    "MiniTransformer (nuestro)": [4, 4, 128, n_param_mini],
    "GPT-2 small español": [cfg.n_layer, cfg.n_head, cfg.n_embd, n_param_gpt],
    "GPT-3": [96, 96, 12288, 175_000_000_000],
}, index=["bloques", "cabezas", "d_model", "parámetros"]).map(lambda v: f"{v:,}")

In [ ]:
# 3.2 Mapas de atención reales de GPT-2
frase = "El perro persiguió al gato porque tenía hambre"
entrada = tokenizer(frase, return_tensors="pt").to(device)
with torch.no_grad():
    salida = modelo(**entrada, output_attentions=True)

atenciones = salida.attentions            # tupla: una por capa → [batch, cabezas, T, T]
toks = [tokenizer.decode([i]).strip() for i in entrada["input_ids"][0]]
print(f"Capas: {len(atenciones)} | forma por capa: {tuple(atenciones[0].shape)}")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (capa, cabeza) in zip(axes, [(0, 0), (5, 3), (11, 7)]):
    plot_atencion(atenciones[capa][0, cabeza], toks, f"Capa {capa} · Cabeza {cabeza}", ax=ax)
plt.tight_layout(); plt.show()

**¿Qué observar?**
- Todos los mapas son **triangulares** → GPT-2 usa máscara causal, igual que la nuestra.
- Muchas cabezas ponen mucho peso en el **primer token** (funciona como "lugar de descanso" cuando no hay nada relevante que mirar).
- Hay cabezas que miran al **token anterior** (diagonal desplazada) y otras que conectan palabras relacionadas (ej. *tenía* → *perro* / *gato*).
- Nadie programó estas reglas: **emergen del entrenamiento**.

---
## ☕ Pausa — 10 minutos
---

## 4. Encoder (BERT) vs Decoder (GPT)

| | **BERT** (Encoder-only) | **GPT** (Decoder-only) |
|---|---|---|
| Atención | **Bidireccional** (ve izquierda y derecha) | **Causal** (solo ve lo anterior) |
| Tarea de entrenamiento | *Masked LM*: adivinar palabras tapadas `[MASK]` | *Causal LM*: predecir el **siguiente** token |
| Sirve para | **Entender**: clasificar, buscar, extraer entidades | **Generar**: chat, resumir, escribir, programar |
| Ejemplos | BERT, BETO, RoBERTa, modelos de embeddings | GPT-2/3/4, Llama, Qwen, Claude |

Probemos ambos con frases en español.

In [ ]:
# 4.1 BERT rellena huecos usando el contexto de AMBOS lados
rellenar = pipeline("fill-mask", model=MODELO_BERT, device=device)

frases_mask = [
    "La capital de Colombia es [MASK].",
    "Me senté en el [MASK] del parque a leer un libro.",
    "Fui al [MASK] a retirar dinero del cajero.",
]
for f in frases_mask:
    preds = rellenar(f, top_k=5)
    print(f"\n{f}")
    for p in preds:
        print(f"   {p['token_str']:<12} {p['score']:.3f}")

In [ ]:
# 4.2 GPT solo puede CONTINUAR el texto (no ve lo que viene después)
for inicio_txt in ["La capital de Colombia es", "Me senté en el"]:
    ent = tokenizer(inicio_txt, return_tensors="pt").to(device)
    with torch.no_grad():
        gen = modelo.generate(**ent, max_new_tokens=15, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    print(f"→ {tokenizer.decode(gen[0], skip_special_tokens=True)}")

### 📚 Teoría: la tercera familia, Encoder-Decoder
El Transformer original de 2017 tenía **las dos partes**: un encoder que lee la frase de entrada (ej. en inglés) y un decoder que genera la salida (ej. en español) mirando al encoder mediante *cross-attention*.
- Ejemplos: **T5**, **BART**, **mBART**, Whisper (voz → texto).
- Buenos para tareas *texto → texto* con entrada y salida bien separadas: traducción, resumen.

**¿Por qué hoy casi todos los LLMs son decoder-only?** Porque *predecir el siguiente token* es una tarea universal: con suficiente escala, el mismo modelo aprende a traducir, resumir, clasificar y conversar, solo cambiando el texto de entrada (el **prompt**). Esa es la base de la Clase 3.

**¿Y BERT murió?** No: los modelos encoder siguen siendo la mejor opción para **embeddings** (búsqueda semántica, RAG) y clasificación barata y rápida.

---
## 5. ¿Cómo genera texto un GPT? Predicción del siguiente token

Un GPT es una función que recibe tokens y devuelve una **distribución de probabilidad sobre el siguiente token**. Generar texto es repetir:

```
prompt → modelo → probabilidades del siguiente token → elegir uno → agregarlo al prompt → repetir
```
Esto se llama generación **autorregresiva**. ChatGPT y Claude escriben así, token por token.

In [ ]:
# 5.1 Las probabilidades del siguiente token
prompt = "La inteligencia artificial es"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    logits = modelo(ids).logits            # [1, T, vocab]

probs = F.softmax(logits[0, -1], dim=-1)   # solo nos interesa la ÚLTIMA posición
top = torch.topk(probs, 10)
tabla_top = pd.DataFrame({
    "token": [tokenizer.decode([i]) for i in top.indices.tolist()],
    "probabilidad": top.values.tolist(),
})
tabla_top.plot.barh(x="token", y="probabilidad", legend=False, color="#7B2CBF", figsize=(7, 4),
                    title=f"Siguiente token después de: '{prompt}'").invert_yaxis()
plt.tight_layout(); plt.show()
tabla_top

In [ ]:
# 5.2 Generación "greedy" hecha a mano: siempre el token más probable
def generar_greedy(prompt, n_tokens=25):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = modelo(ids).logits[0, -1]
        siguiente = torch.argmax(logits).view(1, 1)       # el más probable
        ids = torch.cat([ids, siguiente], dim=1)          # lo agregamos y repetimos
    return tokenizer.decode(ids[0])

print(generar_greedy("La inteligencia artificial es"))

### 🎲 Estrategias de muestreo
*Greedy* tiende a repetirse y a ser aburrido. Los chatbots **muestrean** con tres perillas:

| Parámetro | Qué hace | Efecto |
|---|---|---|
| **temperatura** | divide los logits: `softmax(logits / T)` | T < 1 → más seguro y repetitivo · T > 1 → más creativo y caótico |
| **top-k** | solo considera los *k* tokens más probables | corta la "cola" de tokens absurdos |
| **top-p** (*nucleus*) | considera los tokens más probables hasta que su probabilidad acumulada llega a *p* | tamaño del corte adaptativo |

In [ ]:
# 5.3 Implementamos el muestreo nosotros mismos
def muestrear(logits, temperatura=1.0, top_k=0, top_p=1.0):
    logits = logits / max(temperatura, 1e-6)
    if top_k > 0:                                          # top-k: anulamos todo lo que no esté en el top k
        umbral = torch.topk(logits, top_k).values[-1]
        logits = logits.masked_fill(logits < umbral, float("-inf"))
    if top_p < 1.0:                                        # top-p: núcleo de probabilidad acumulada
        orden, indices = torch.sort(logits, descending=True)
        acumulada = torch.cumsum(F.softmax(orden, dim=-1), dim=-1)
        quitar = acumulada > top_p
        quitar[1:] = quitar[:-1].clone()                   # conservamos el token que cruza el umbral
        quitar[0] = False
        logits[indices[quitar]] = float("-inf")
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)         # sorteo según las probabilidades

def generar_manual(prompt, n_tokens=30, **kwargs):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = modelo(ids).logits[0, -1]
        ids = torch.cat([ids, muestrear(logits, **kwargs).view(1, 1)], dim=1)
    return tokenizer.decode(ids[0])

fijar_semilla()
for config in [dict(temperatura=0.3), dict(temperatura=1.0, top_k=50), dict(temperatura=1.5, top_p=0.9)]:
    print(f"{config}\n → {generar_manual('La inteligencia artificial es', **config)}\n")

In [ ]:
# 5.4 Visualizar el efecto de la temperatura sobre la distribución
logits_ult = logits[0, -1]
top_ids = torch.topk(logits_ult, 8).indices
etiquetas = [tokenizer.decode([i]).strip() or repr(tokenizer.decode([i])) for i in top_ids.tolist()]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for ax, temp in zip(axes, [0.5, 1.0, 1.8]):
    p = F.softmax(logits_ult / temp, dim=-1)[top_ids].cpu()
    ax.bar(etiquetas, p, color="#7B2CBF")
    ax.set_title(f"temperatura = {temp}")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("probabilidad")
plt.tight_layout(); plt.show()

In [ ]:
# 5.5 Lo mismo con la función oficial: model.generate hace por dentro lo que acabamos de programar
ent = tokenizer("La inteligencia artificial es", return_tensors="pt").to(device)
fijar_semilla()
with torch.no_grad():
    gen = modelo.generate(**ent, max_new_tokens=30, do_sample=True, temperature=0.8, top_k=50, top_p=0.95,
                          repetition_penalty=1.2, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(gen[0], skip_special_tokens=True))

### 📚 Teoría: detalles de generación que usan los LLMs reales
- **KV-cache:** en nuestro bucle manual recalculamos *toda* la secuencia en cada paso. `generate` guarda las K y V ya calculadas y solo procesa el token nuevo → mucho más rápido. Por eso la memoria de un LLM crece con la longitud de la conversación.
- **repetition_penalty:** baja el logit de los tokens que ya aparecieron (evita bucles "es es es…").
- **Beam search:** explora varias secuencias candidatas en paralelo y se queda con la más probable. Útil en traducción; en chat suele dar textos sosos.
- **¿Por qué el mismo prompt da respuestas distintas en ChatGPT?** Porque hay muestreo (temperatura > 0). Con `do_sample=False` la salida es determinista.
- **Valores típicos:** chat general T≈0.7, top-p≈0.9 · código o datos T≈0–0.3 · escritura creativa T≈1.0.

---
## 6. MiniGPT: de modelo base a chatbot (fine-tuning supervisado)

GPT-2 aprendió a **continuar texto de Wikipedia**, no a **conversar**. Si le damos un formato de chat, no sabe qué hacer.
Le enseñaremos un formato fijo:

```
Eres MiniGPT, un asistente que responde en español de forma breve y clara.

### Usuario: ¿Qué es un token?
### MiniGPT: Un token es un pedazo de texto...<|endoftext|>
```
Primero veamos cómo responde **antes** de entrenar.

In [ ]:
# 6.1 Formato de chat + función para responder (con criterio de parada)
def construir_prompt(pregunta):
    return f"{SYSTEM_PROMPT}\n\n{TAG_USUARIO} {pregunta}\n{TAG_BOT}"

class PararEnTag(StoppingCriteria):
    # Detiene la generación si el modelo empieza a escribir el turno del usuario
    def __init__(self, tokenizer, tag, n_prompt):
        self.tokenizer, self.tag, self.n_prompt = tokenizer, tag, n_prompt
    def __call__(self, input_ids, scores, **kwargs):
        texto = self.tokenizer.decode(input_ids[0, self.n_prompt:])
        return torch.full((input_ids.shape[0],), self.tag in texto, dtype=torch.bool, device=input_ids.device)

def responder(modelo, pregunta, max_new_tokens=60, muestrear=False, temperatura=0.7, top_p=0.9):
    # muestrear=False → greedy: siempre el token más probable (ideal para ver qué APRENDIÓ el modelo)
    entrada = tokenizer(construir_prompt(pregunta), return_tensors="pt").to(device)
    n_prompt = entrada["input_ids"].shape[1]
    opciones = dict(do_sample=True, temperature=temperatura, top_p=top_p) if muestrear else dict(do_sample=False)
    with torch.no_grad():
        salida = modelo.generate(
            **entrada, max_new_tokens=max_new_tokens, **opciones,
            no_repeat_ngram_size=3,                    # evita bucles sin castigar las palabras del prompt
            pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
            stopping_criteria=StoppingCriteriaList([PararEnTag(tokenizer, "###", n_prompt)]),
        )
    texto = tokenizer.decode(salida[0, n_prompt:], skip_special_tokens=True)
    return texto.split("###")[0].split("\n")[0].strip()       # nos quedamos con la primera línea

preguntas_prueba = ["¿Cómo te llamas?", "¿Qué es un token?", "¿Qué es el overfitting?"]
preguntas_nuevas = ["¿Me dices tu nombre?", "¿Qué es una red convolucional?", "¿Para qué sirve el learning rate?",
                    "¿Cuál es la capital de Francia?"]
antes = {p: responder(modelo, p) for p in preguntas_prueba + preguntas_nuevas}
for p in preguntas_prueba:
    print(f"🧑 {p}\n🤖 {antes[p]}\n")

😅 El modelo base **no sigue el formato**: continúa con cualquier texto parecido a Wikipedia.
Para arreglarlo hacemos **Supervised Fine-Tuning (SFT)**: le mostramos ~90 ejemplos de preguntas y respuestas con el formato correcto. Es exactamente el paso que OpenAI hizo para pasar de GPT-3 a ChatGPT, pero con millones de ejemplos.

In [ ]:
# 6.2 Cargar el dataset de diálogos
df = pd.read_json(RUTA_DATOS, lines=True)
print(f"{len(df)} diálogos")

df_mezcla = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_val = 8
df_val, df_train = df_mezcla.iloc[:n_val], df_mezcla.iloc[n_val:]
print(f"train: {len(df_train)} | validación: {len(df_val)}")
df.head(8)

### 🎯 El truco del SFT: solo calculamos la pérdida sobre la respuesta
No queremos que el modelo aprenda a escribir el *system prompt* ni la pregunta (eso lo pone el usuario), solo **a responder**.
Por eso, en las etiquetas (`labels`) ponemos **-100** en los tokens del prompt: PyTorch **ignora** esas posiciones al calcular la pérdida.

In [ ]:
# 6.3 Tokenizar: prompt + respuesta + EOS, con labels = -100 en la parte del prompt
MAX_LEN = 160

def codificar(pregunta, respuesta):
    ids_prompt = tokenizer.encode(construir_prompt(pregunta))
    ids_resp = tokenizer.encode(" " + respuesta) + [tokenizer.eos_token_id]
    ids = (ids_prompt + ids_resp)[:MAX_LEN]
    labels = ([-100] * len(ids_prompt) + ids_resp)[:MAX_LEN]
    return {"input_ids": ids, "labels": labels}

datos_train = [codificar(p, r) for p, r in zip(df_train.pregunta, df_train.respuesta)]
datos_val = [codificar(p, r) for p, r in zip(df_val.pregunta, df_val.respuesta)]

# Visualizamos un ejemplo: en gris lo que se ignora (-100), en claro lo que el modelo debe aprender
ej = datos_train[0]
ignorado = tokenizer.decode([i for i, l in zip(ej["input_ids"], ej["labels"]) if l == -100])
aprende = tokenizer.decode([i for i, l in zip(ej["input_ids"], ej["labels"]) if l != -100])
print("IGNORADO (-100):", repr(ignorado))
print("APRENDE        :", repr(aprende))
print("Longitud máxima en train:", max(len(d["input_ids"]) for d in datos_train), "tokens")

In [ ]:
# 6.4 Batches con padding dinámico (rellenamos hasta el más largo de cada batch)
def collate(batch):
    largo = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), largo), tokenizer.pad_token_id)
    labels = torch.full((len(batch), largo), -100)
    mask = torch.zeros((len(batch), largo), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        labels[i, :n] = torch.tensor(b["labels"])
        mask[i, :n] = 1
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}

loader_train = torch.utils.data.DataLoader(datos_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
loader_val = torch.utils.data.DataLoader(datos_val, batch_size=BATCH_SIZE, collate_fn=collate)
lote = next(iter(loader_train))
print({k: tuple(v.shape) for k, v in lote.items()})

In [ ]:
# 6.5 Congelar las primeras capas (entrena más rápido en CPU y "olvida" menos lo que ya sabía)
for p in modelo.parameters():
    p.requires_grad = True
if CAPAS_CONGELADAS:
    for p in modelo.transformer.wte.parameters():  p.requires_grad = False   # embeddings de tokens
    for p in modelo.transformer.wpe.parameters():  p.requires_grad = False   # embeddings de posición
    for bloque in modelo.transformer.h[:CAPAS_CONGELADAS]:
        for p in bloque.parameters():
            p.requires_grad = False

entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Bloques congelados: {CAPAS_CONGELADAS}/{cfg.n_layer}")
print(f"Parámetros entrenables: {entrenables:,} de {n_param_gpt:,} ({entrenables / n_param_gpt:.0%})")

In [ ]:
# 6.6 Bucle de entrenamiento (el mismo esquema de la Unidad 1: forward → loss → backward → step)
@torch.no_grad()
def evaluar(loader):
    modelo.eval()
    perdidas = [modelo(**{k: v.to(device) for k, v in b.items()}).loss.item() for b in loader]
    return sum(perdidas) / len(perdidas)

optimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=0.01)
pasos_totales = EPOCHS * len(loader_train)
scheduler = get_linear_schedule_with_warmup(optimizador, num_warmup_steps=int(0.1 * pasos_totales),
                                            num_training_steps=pasos_totales)

historial = {"train": [], "val": [evaluar(loader_val)]}
print(f"Pérdida de validación ANTES de entrenar: {historial['val'][0]:.3f}")
fijar_semilla()
inicio = time.time()
for epoca in range(1, EPOCHS + 1):
    modelo.train()
    perdidas = []
    for lote in loader_train:
        lote = {k: v.to(device) for k, v in lote.items()}
        perdida = modelo(**lote).loss          # Hugging Face calcula la cross-entropy desplazando los labels
        perdida.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        optimizador.step(); scheduler.step(); optimizador.zero_grad()
        perdidas.append(perdida.item())
    historial["train"].append(sum(perdidas) / len(perdidas))
    historial["val"].append(evaluar(loader_val))
    print(f"Época {epoca}/{EPOCHS} | train {historial['train'][-1]:.3f} | val {historial['val'][-1]:.3f} | {time.time() - inicio:.0f}s")
modelo.eval();

In [ ]:
# 6.7 Curvas de pérdida
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), historial["train"], "o-", label="train", color="#7B2CBF")
plt.plot(range(0, EPOCHS + 1), historial["val"], "s--", label="validación", color="#E0218A")
plt.xlabel("época"); plt.ylabel("pérdida (cross-entropy)"); plt.title("Fine-tuning de MiniGPT")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# 6.8 ¿Aprendió? Mismas preguntas de antes + preguntas NUEVAS que no estaban en el dataset
filas = []
for p in preguntas_prueba + preguntas_nuevas:
    filas.append({"pregunta": p, "¿en el dataset?": "sí" if p in set(df.pregunta) else "NO",
                  "antes (GPT-2 base)": antes.get(p, "—"), "después (MiniGPT)": responder(modelo, p)})
pd.DataFrame(filas)

In [ ]:
# 6.9 Guardar MiniGPT (lo usaremos en la Clase 3)
DIR_SALIDA.mkdir(parents=True, exist_ok=True)
modelo.save_pretrained(DIR_SALIDA)
tokenizer.save_pretrained(DIR_SALIDA)
print("✓ MiniGPT guardado en:", DIR_SALIDA.resolve())

In [ ]:
# 6.10 ¡A chatear! (escribe 'salir' para terminar)
def chatear():
    print("MiniGPT listo. Escribe 'salir' para terminar.\n")
    while True:
        pregunta = input("🧑 Tú: ").strip()
        if pregunta.lower() in {"salir", "exit", "quit", ""}:
            print("🤖 MiniGPT: ¡Hasta luego!")
            break
        print("🤖 MiniGPT:", responder(modelo, pregunta), "\n")

# chatear()   # ← quita el comentario para conversar

---
## ✅ Cierre

**Lo que construimos hoy**
1. Vimos que los LLMs leen **tokens subword**, no palabras.
2. Programamos **self-attention** (Q, K, V, softmax, máscara causal), **multi-cabeza** y un **bloque Transformer**.
3. Vimos **BERT** (entiende, bidireccional) vs **GPT** (genera, causal).
4. Programamos la **generación autorregresiva** con temperatura, top-k y top-p.
5. Convertimos GPT-2 en **MiniGPT** con *fine-tuning supervisado*.

**Limitaciones de MiniGPT** (¡y lo que resolveremos en la Clase 3!)
| Problema | Por qué | Solución en la Clase 3 |
|---|---|---|
| Solo sabe lo de sus ~90 ejemplos; inventa en lo demás | Modelo pequeño + pocos datos | Usar un **LLM instruct** (Qwen3) entrenado con millones de instrucciones |
| No recuerda la conversación | Cada pregunta se envía sola | **Memoria**: reenviar el historial con un *chat template* |
| No se le pueden dar instrucciones nuevas | Solo aprendió un formato | **Prompt engineering**: system prompt, few-shot, chain-of-thought |

### 🏋️ Reto / tarea
1. Agrega **20 diálogos nuevos** al JSONL sobre un tema que te guste (fútbol, cocina, tu empresa…) y vuelve a entrenar. ¿Responde bien preguntas parecidas pero no idénticas?
2. Cambia `CAPAS_CONGELADAS` a 0 y a 10. ¿Qué pasa con el tiempo de entrenamiento y con las respuestas?
3. Prueba `responder()` con temperatura 0.2 y 1.3. Describe la diferencia.

### 🧑‍🏫 Preguntas frecuentes de estudiantes (con respuesta)
- **"¿Por qué la pérdida de validación es tan alta comparada con train?"** → Con ~85 ejemplos el modelo memoriza; la validación tiene preguntas que nunca vio. Es *overfitting* visible: buen momento para recordar la Unidad 1.
- **"¿Por qué responde mal 'la capital de Francia'?"** → No estaba en el dataset de SFT y GPT-2 small tiene poco conocimiento. El SFT **enseña formato y estilo**, no conocimiento nuevo masivo. El conocimiento viene del preentrenamiento (por eso los LLMs grandes se preentrenan con billones de tokens).
- **"¿ChatGPT se entrena así?"** → Sí en espíritu: 1) preentrenamiento (predecir el siguiente token en internet), 2) SFT con ejemplos de conversación escritos por personas, 3) RLHF/DPO para preferir respuestas útiles y seguras. Hoy hicimos el paso 2 en miniatura.
- **"¿Por qué -100?"** → Es el `ignore_index` por defecto de `CrossEntropyLoss` en PyTorch.
- **"¿Por qué congelar capas?"** → Las primeras capas capturan patrones generales del idioma; ajustar solo las últimas es más rápido y reduce el *olvido catastrófico*. En LLMs grandes se usa **LoRA** (Clase 4): entrenar matrices pequeñas añadidas en vez de los pesos originales.
- **"¿Puedo usar GPU?"** → Sí: el notebook detecta `cuda`/`mps` y entonces entrena todas las capas (`CAPAS_CONGELADAS = 0`).